# Análisis de Datos: Diabetes — Exploración y Modelado Básico

**Alumno:** Zamudio Damián Oscar Kuricaveri — 22120729  
**Materia:** Recuperación de Información  
**ITM Morelia — 2026**

---

## Objetivo

Explorar un dataset de diabetes simulado, identificar factores de riesgo relevantes, aplicar limpieza de datos, generar visualizaciones descriptivas y entrenar un clasificador básico.

**Variables del dataset:**
- `glucosa`: nivel de glucosa en sangre (mg/dL)
- `presion`: presión arterial diastólica (mmHg)
- `imc`: índice de masa corporal
- `edad`: edad del paciente
- `insulina`: nivel de insulina (μU/ml)
- `historial_familiar`: antecedentes familiares de diabetes (0/1)
- `actividad_fisica`: horas de ejercicio por semana
- `diagnostico`: resultado (0=No diabético, 1=Diabético)

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(42)
print('Librerías cargadas correctamente')

## 2. Generación del dataset simulado

In [ ]:
n = 500

# Pacientes sanos (no diabéticos)
n_sanos = 300
sanos = pd.DataFrame({
    'glucosa': np.random.normal(95, 15, n_sanos).clip(70, 140),
    'presion': np.random.normal(72, 8, n_sanos).clip(50, 100),
    'imc': np.random.normal(24, 3.5, n_sanos).clip(17, 38),
    'edad': np.random.randint(20, 65, n_sanos),
    'insulina': np.random.normal(80, 25, n_sanos).clip(15, 200),
    'historial_familiar': np.random.choice([0, 1], n_sanos, p=[0.75, 0.25]),
    'actividad_fisica': np.random.normal(4.5, 1.5, n_sanos).clip(0, 10),
    'diagnostico': 0,
})

# Pacientes diabéticos
n_diab = 200
diabeticos = pd.DataFrame({
    'glucosa': np.random.normal(150, 30, n_diab).clip(100, 250),
    'presion': np.random.normal(82, 10, n_diab).clip(60, 120),
    'imc': np.random.normal(31, 5, n_diab).clip(22, 50),
    'edad': np.random.randint(35, 80, n_diab),
    'insulina': np.random.normal(140, 50, n_diab).clip(50, 350),
    'historial_familiar': np.random.choice([0, 1], n_diab, p=[0.4, 0.6]),
    'actividad_fisica': np.random.normal(2.0, 1.5, n_diab).clip(0, 7),
    'diagnostico': 1,
})

df = pd.concat([sanos, diabeticos], ignore_index=True).sample(frac=1, random_state=42)
df = df.round(2)

print(f'Dataset generado: {df.shape}')
print(f'Distribución del diagnóstico:')
print(df['diagnostico'].value_counts())
df.head()

## 3. Exploración inicial

In [ ]:
print('=== ESTADÍSTICAS GENERALES ===')
print(df.describe().round(2))

print('\n=== VALORES NULOS ===')
print(df.isnull().sum())

print('\n=== CORRELACIÓN CON DIAGNÓSTICO ===')
corr = df.corr()['diagnostico'].sort_values(ascending=False)
print(corr.round(3))

## 4. Limpieza básica

In [ ]:
# Verificar valores fuera de rango clínico
print('Valores extremos detectados:')
print(f"  Glucosa > 200: {(df['glucosa'] > 200).sum()}")
print(f"  IMC > 45: {(df['imc'] > 45).sum()}")
print(f"  Presión > 110: {(df['presion'] > 110).sum()}")

# Agregar categorías de IMC
def categoria_imc(imc):
    if imc < 18.5: return 'Bajo peso'
    elif imc < 25: return 'Normal'
    elif imc < 30: return 'Sobrepeso'
    else: return 'Obesidad'

df['categoria_imc'] = df['imc'].apply(categoria_imc)
print('\nDistribución por categoría de IMC:')
print(df['categoria_imc'].value_counts())

## 5. Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Análisis Exploratorio — Dataset Diabetes (Simulado)', fontsize=14, fontweight='bold')

colores = {0: '#2ecc71', 1: '#e74c3c'}
etiquetas = {0: 'No Diabético', 1: 'Diabético'}

# 1. Distribución de glucosa
ax = axes[0, 0]
for diag in [0, 1]:
    subset = df[df['diagnostico'] == diag]['glucosa']
    ax.hist(subset, bins=25, alpha=0.6, color=colores[diag], label=etiquetas[diag], edgecolor='white')
ax.set_title('Distribución de Glucosa')
ax.set_xlabel('Glucosa (mg/dL)')
ax.legend()

# 2. IMC vs Glucosa (scatter)
ax = axes[0, 1]
for diag in [0, 1]:
    subset = df[df['diagnostico'] == diag]
    ax.scatter(subset['imc'], subset['glucosa'],
               c=colores[diag], label=etiquetas[diag], alpha=0.5, s=20)
ax.set_title('IMC vs Glucosa')
ax.set_xlabel('IMC')
ax.set_ylabel('Glucosa (mg/dL)')
ax.legend()

# 3. Distribución por categoría IMC
ax = axes[0, 2]
imc_diab = df[df['diagnostico'] == 1]['categoria_imc'].value_counts()
ax.bar(imc_diab.index, imc_diab.values, color='#e74c3c', edgecolor='white')
ax.set_title('Distribución IMC en Diabéticos')
ax.set_xlabel('Categoría IMC')

# 4. Edad por diagnóstico (boxplot)
ax = axes[1, 0]
datos_edad = [df[df['diagnostico'] == d]['edad'].values for d in [0, 1]]
bp = ax.boxplot(datos_edad, labels=['No Diabético', 'Diabético'], patch_artist=True)
bp['boxes'][0].set_facecolor('#2ecc71')
bp['boxes'][1].set_facecolor('#e74c3c')
ax.set_title('Distribución de Edad por Diagnóstico')
ax.set_ylabel('Edad')

# 5. Historial familiar vs diagnóstico
ax = axes[1, 1]
tabla = df.groupby(['historial_familiar', 'diagnostico']).size().unstack()
tabla.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c'], edgecolor='white')
ax.set_title('Historial Familiar vs Diagnóstico')
ax.set_xlabel('Historial Familiar (0=No, 1=Sí)')
ax.legend(['No Diabético', 'Diabético'])
ax.tick_params(axis='x', rotation=0)

# 6. Correlación con diagnóstico
ax = axes[1, 2]
corr = df.corr()['diagnostico'].drop('diagnostico').sort_values()
colors = ['#e74c3c' if c > 0 else '#2ecc71' for c in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor='white')
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Correlación Variables — Diagnóstico')
ax.set_xlabel('Coeficiente de correlación')

plt.tight_layout()
plt.savefig('assets/diabetes_analisis.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Modelo básico de clasificación

In [ ]:
features = ['glucosa', 'presion', 'imc', 'edad', 'insulina', 'historial_familiar', 'actividad_fisica']
X = df[features]
y = df['diagnostico']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

modelo = LogisticRegression(random_state=42, max_iter=1000)
modelo.fit(X_train_sc, y_train)

y_pred = modelo.predict(X_test_sc)

print('=== REPORTE DE CLASIFICACIÓN ===')
print(classification_report(y_test, y_pred, target_names=['No Diabético', 'Diabético']))

print('=== MATRIZ DE CONFUSIÓN ===')
cm = confusion_matrix(y_test, y_pred)
print(cm)

print('\n=== IMPORTANCIA DE CARACTERÍSTICAS (coeficientes) ===')
coefs = pd.Series(modelo.coef_[0], index=features).sort_values(ascending=False)
print(coefs.round(4))

## 7. Conclusiones

1. **Variables más predictivas:** La glucosa y el IMC presentaron la mayor correlación con el diagnóstico positivo, lo que concuerda con la evidencia clínica sobre factores de riesgo de diabetes tipo 2.

2. **Actividad física como factor protector:** La actividad física mostró correlación negativa con el diagnóstico, siendo el factor protector más relevante en el dataset.

3. **Historial familiar:** Los pacientes con antecedentes familiares presentaron mayor prevalencia de diagnóstico positivo, confirmando el componente hereditario de la enfermedad.

4. **Modelo de clasificación:** La Regresión Logística alcanzó un F1-score razonable para un modelo base, siendo la glucosa y el IMC los coeficientes más altos en el modelo entrenado.

5. **Limitaciones:** Los datos son simulados. En un análisis real, se requeriría validación clínica, manejo de valores faltantes (frecuentes en datos médicos reales) y consideraciones éticas sobre el uso de datos sensibles de salud.

---
*Notebook generado con fines académicos — ITM Morelia 2026*